# 🔴 EducaStock — Treinamento do Classificador de Risco de Vencimento (TFLite)

> **ML Caso de Uso 2** — ONG Casa da Criança  
> Modelo: Random Forest → destilado para TFLite (roda on-device no app Flutter)  
> Fallback automático: Classificador baseado em regras (sem ML)  

## O que este notebook faz
1. Autentica com Firebase Admin SDK
2. Exporta lotes (`batches`) do Firestore com campos de validade e quantidade
3. Calcula as 4 features normalizadas por lote
4. Rotula automaticamente via mesmas regras do app (RuleBasedRiskClassifier)
5. **Se houver poucos lotes reais** → aumenta com dados sintéticos realistas
6. Treina Random Forest + avalia métricas
7. Destila para rede neural densa → converte para TFLite (formato on-device)
8. **Baixa `expiry_risk.tflite`** → você coloca em `assets/models/` e rebuild

## Features de entrada (4 valores, todos normalizados [0, 1])
| # | Feature | Descrição |
|---|---------|----------|
| 0 | `days_to_expiry_norm` | dias até vencimento ÷ 365 (1.0 = sem validade) |
| 1 | `quantity_ratio` | quantidade atual ÷ quantidade inicial |
| 2 | `days_since_entry_norm` | dias em estoque ÷ 365 |
| 3 | `is_no_expiry` | 1.0 = sem validade, 0.0 = perecível |

## Saída (3 classes)
| Índice | Label | Significado |
|--------|-------|------------|
| 0 | `verde` | Seguro |
| 1 | `amarelo` | Atenção — vencimento próximo |
| 2 | `vermelho` | Crítico — distribuição urgente |

## Pré-requisitos
- Arquivo `serviceAccountKey.json` do Firebase Admin (upload na próxima célula)
- Para gerar: Firebase Console → Configurações → Contas de serviço → Gerar nova chave privada


In [ ]:
# ─── Instalar dependências ─────────────────────────────────────────────────
!pip install firebase-admin pandas numpy scikit-learn tensorflow --quiet
print("✅ Dependências instaladas")

In [ ]:
# ─── Upload do arquivo de credenciais Firebase ─────────────────────────────
from google.colab import files

print("Faça upload do serviceAccountKey.json")
print("Caminho: Firebase Console → Configurações do Projeto → Contas de Serviço → Gerar nova chave privada")
uploaded = files.upload()
SERVICE_ACCOUNT_KEY = list(uploaded.keys())[0]
print(f"\n✅ Arquivo carregado: {SERVICE_ACCOUNT_KEY}")

In [ ]:
# ─── Inicializar Firebase Admin ────────────────────────────────────────────
import firebase_admin
from firebase_admin import credentials, firestore

if not firebase_admin._apps:
    cred = credentials.Certificate(SERVICE_ACCOUNT_KEY)
    firebase_admin.initialize_app(cred)

db = firestore.client()
print("✅ Firebase Admin inicializado")

In [ ]:
# ─── Exportar lotes do Firestore ───────────────────────────────────────────
import pandas as pd
from datetime import datetime, timezone

def export_batches():
    """Exporta a coleção 'batches' do Firestore como DataFrame."""
    docs = db.collection('batches').stream()
    rows = []
    for doc in docs:
        d = doc.to_dict()
        d['id'] = doc.id
        rows.append(d)
    return pd.DataFrame(rows)

df_raw = export_batches()
print(f"📦 Lotes encontrados: {len(df_raw)}")
if len(df_raw) > 0:
    print(f"   Colunas: {list(df_raw.columns)}")
    print(f"\nPrimeiros registros:")
    print(df_raw[['productName','quantity','initialQuantity','expiryDate','noExpiry','status']].head())
else:
    print("⚠️  Nenhum lote encontrado. O modelo será treinado apenas com dados sintéticos.")

In [ ]:
# ─── Extrair features dos lotes reais ─────────────────────────────────────
import numpy as np

MAX_DAYS = 365.0
MIN_REAL_BATCHES = 30  # mínimo de lotes reais para evitar overfitting

def parse_date(val):
    """Converte string ISO ou Timestamp do Firestore para datetime."""
    if val is None:
        return None
    if hasattr(val, 'timestamp'):  # Firestore Timestamp
        return val.ToDatetime().replace(tzinfo=timezone.utc)
    if isinstance(val, str):
        return datetime.fromisoformat(val.replace('Z', '+00:00'))
    return None

def label_from_features(days_to_expiry, quantity_ratio, days_since_entry_norm, is_no_expiry):
    """Mesma lógica do RuleBasedRiskClassifier no app Flutter."""
    if is_no_expiry:
        return 0  # verde
    if days_to_expiry <= 0:
        return 2  # vermelho — vencido
    if days_to_expiry <= 7:
        return 2  # vermelho — crítico
    if days_to_expiry <= 30:
        if quantity_ratio > 0.8 and days_to_expiry <= 20:
            return 2  # vermelho — lento e prestes a vencer
        return 1  # amarelo
    # Verde com possível alerta de giro baixo
    stale = days_since_entry_norm > (60 / MAX_DAYS) and quantity_ratio > 0.8
    if stale and days_to_expiry <= 90:
        return 1  # amarelo — estoque parado
    return 0  # verde

def extract_features(df):
    """Extrai e normaliza as 4 features de cada lote."""
    now = datetime.now(timezone.utc)
    X, y = [], []

    for _, row in df.iterrows():
        no_expiry = bool(row.get('noExpiry', False))
        expiry_date = parse_date(row.get('expiryDate'))
        entry_date = parse_date(row.get('entryDate'))
        qty = float(row.get('quantity', 0))
        init_qty = float(row.get('initialQuantity', qty))
        if init_qty <= 0:
            init_qty = max(qty, 1)

        # Feature 0: days_to_expiry_norm
        if no_expiry or expiry_date is None:
            days_to_expiry = MAX_DAYS
            days_to_expiry_norm = 1.0
        else:
            days_to_expiry = (expiry_date - now).days
            days_to_expiry_norm = float(np.clip(days_to_expiry / MAX_DAYS, 0.0, 1.0))

        # Feature 1: quantity_ratio
        quantity_ratio = float(np.clip(qty / init_qty, 0.0, 1.0))

        # Feature 2: days_since_entry_norm
        if entry_date is None:
            days_since_entry_norm = 0.0
        else:
            days_since_entry = (now - entry_date).days
            days_since_entry_norm = float(np.clip(days_since_entry / MAX_DAYS, 0.0, 1.0))

        # Feature 3: is_no_expiry
        is_no_expiry = 1.0 if no_expiry else 0.0

        features = [days_to_expiry_norm, quantity_ratio, days_since_entry_norm, is_no_expiry]
        label = label_from_features(
            days_to_expiry=int(days_to_expiry),
            quantity_ratio=quantity_ratio,
            days_since_entry_norm=days_since_entry_norm,
            is_no_expiry=bool(no_expiry)
        )

        X.append(features)
        y.append(label)

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)


if len(df_raw) > 0:
    X_real, y_real = extract_features(df_raw)
    print(f"✅ Features extraídas de {len(X_real)} lotes reais")
    print(f"   Distribuição de classes: {dict(zip(*np.unique(y_real, return_counts=True)))}")
    print(f"   (0=verde, 1=amarelo, 2=vermelho)")
else:
    X_real = np.empty((0, 4), dtype=np.float32)
    y_real = np.empty(0, dtype=np.int32)
    print("Nenhum dado real. Usando apenas sintético.")

In [ ]:
# ─── Gerar dados sintéticos (complementa os reais se necessário) ───────────

RANDOM_SEED = 42
TARGET_SAMPLES = max(4000, len(X_real) * 10)  # 10x os dados reais, mínimo 4000

def generate_synthetic(n_samples: int, seed: int = 42):
    """Gera amostras sintéticas com as mesmas regras do classificador."""
    rng = np.random.default_rng(seed)
    X, y = [], []

    for _ in range(n_samples):
        is_no_expiry = bool(rng.choice([False, True], p=[0.65, 0.35]))

        if is_no_expiry:
            days_to_expiry_norm = 1.0
            days_to_expiry = int(MAX_DAYS)
        else:
            days_to_expiry = int(rng.integers(0, int(MAX_DAYS) + 1))
            days_to_expiry_norm = float(days_to_expiry) / MAX_DAYS

        quantity_ratio = float(rng.uniform(0.05, 1.0))
        days_since_entry_norm = float(rng.uniform(0.0, 1.0))
        is_no_expiry_f = 1.0 if is_no_expiry else 0.0

        label = label_from_features(
            days_to_expiry=days_to_expiry,
            quantity_ratio=quantity_ratio,
            days_since_entry_norm=days_since_entry_norm,
            is_no_expiry=is_no_expiry
        )

        # Ruído realista (~5%) para melhor generalização
        if rng.random() < 0.05:
            label = int(rng.integers(0, 3))

        X.append([days_to_expiry_norm, quantity_ratio, days_since_entry_norm, is_no_expiry_f])
        y.append(label)

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)


n_synth = max(TARGET_SAMPLES - len(X_real), 500)  # sempre pelo menos 500 sintéticos
X_synth, y_synth = generate_synthetic(n_synth, seed=RANDOM_SEED)

# Combina dados reais + sintéticos
if len(X_real) > 0:
    X_all = np.vstack([X_real, X_synth])
    y_all = np.concatenate([y_real, y_synth])
else:
    X_all = X_synth
    y_all = y_synth

print(f"📊 Dataset final: {len(X_all)} amostras")
print(f"   Reais: {len(X_real)} | Sintéticos: {len(X_synth)}")
unique, counts = np.unique(y_all, return_counts=True)
labels_map = {0: 'verde', 1: 'amarelo', 2: 'vermelho'}
for u, c in zip(unique, counts):
    print(f"   {labels_map[u]}: {c} ({100*c/len(y_all):.1f}%)")

In [ ]:
# ─── Treinar Random Forest ─────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=RANDOM_SEED, stratify=y_all
)

clf = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    class_weight='balanced',
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = (y_pred == y_test).mean()
print(f"\n=== Random Forest — Acurácia: {acc:.4f} ===")
print(classification_report(y_test, y_pred, target_names=['verde', 'amarelo', 'vermelho']))

# Matriz de confusão
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['verde', 'amarelo', 'vermelho'],
            yticklabels=['verde', 'amarelo', 'vermelho'], ax=ax)
ax.set_xlabel('Previsto')
ax.set_ylabel('Real')
ax.set_title('Matriz de Confusão — Risco de Vencimento')
plt.tight_layout()
plt.show()

# Importância das features
fig2, ax2 = plt.subplots(figsize=(6, 3))
feature_names = ['days_to_expiry_norm', 'quantity_ratio', 'days_since_entry_norm', 'is_no_expiry']
importances = clf.feature_importances_
ax2.barh(feature_names, importances, color='steelblue')
ax2.set_xlabel('Importância')
ax2.set_title('Importância das Features')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Destilar Random Forest → Rede Neural densa → TFLite ──────────────────
# Esta etapa é necessária porque TFLite não suporta RandomForest diretamente.
# Estratégia: treinar uma rede pequena (teacher: RF, student: NN)
import tensorflow as tf

print("Iniciando destilação RF → NN densa...")

# Dataset de destilação com probabilidades suavizadas do RF
N_DISTIL = max(12_000, len(X_all) * 2)
rng_d = np.random.default_rng(RANDOM_SEED + 99)
X_distil = rng_d.random((N_DISTIL, 4)).astype(np.float32)
X_distil[:, 3] = (rng_d.random(N_DISTIL) > 0.65).astype(np.float32)  # ~35% sem validade
y_soft = clf.predict_proba(X_distil).astype(np.float32)  # soft labels do RF

# Rede pequena compatível com TFLite: 4 → 64 → 32 → 3
inputs = tf.keras.Input(shape=(4,), name='features')
x = tf.keras.layers.Dense(64, activation='relu')(inputs)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dense(32, activation='relu')(x)
outputs = tf.keras.layers.Dense(3, activation='softmax', name='risk_probs')(x)

nn_model = tf.keras.Model(inputs, outputs)
nn_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

history = nn_model.fit(
    X_distil, y_soft,
    epochs=80,
    batch_size=128,
    validation_split=0.1,
    verbose=1,
)

# Validação final no conjunto de teste original (labels hard)
y_nn_probs = nn_model.predict(X_test, verbose=0)
y_nn_pred = np.argmax(y_nn_probs, axis=1)
nn_acc = (y_nn_pred == y_test).mean()
print(f"\n✅ Rede Neural destilada — Acurácia no teste: {nn_acc:.4f}")
print(classification_report(y_test, y_nn_pred, target_names=['verde', 'amarelo', 'vermelho']))

In [ ]:
# ─── Converter para TFLite e baixar ───────────────────────────────────────
from google.colab import files

# Converter
converter = tf.lite.TFLiteConverter.from_keras_model(nn_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

# Salvar localmente no Colab
OUTPUT_FILE = 'expiry_risk.tflite'
with open(OUTPUT_FILE, 'wb') as f:
    f.write(tflite_model)

size_kb = len(tflite_model) / 1024
print(f"\n✅ TFLite gerado: {OUTPUT_FILE}")
print(f"   Tamanho: {size_kb:.1f} KB")
print(f"\n🔽 Baixando arquivo...")

# Baixar para o computador
files.download(OUTPUT_FILE)

print("\n📋 PRÓXIMOS PASSOS:")
print("   1. Mova o arquivo baixado para: assets/models/expiry_risk.tflite")
print("   2. Execute: flutter build apk  (ou run)")
print("   3. O app passará a usar o modelo treinado com seus dados reais!")

In [ ]:
# ─── (Opcional) Salvar metadados do modelo ─────────────────────────────────
import json

meta = {
    'model_type': 'random_forest_distilled_tflite',
    'trained_at': datetime.now().isoformat(),
    'real_batches': int(len(X_real)),
    'synthetic_samples': int(len(X_synth)),
    'total_samples': int(len(X_all)),
    'rf_accuracy': float(acc),
    'nn_accuracy': float(nn_acc),
    'features': [
        'days_to_expiry_norm',
        'quantity_ratio',
        'days_since_entry_norm',
        'is_no_expiry',
    ],
    'labels': ['verde', 'amarelo', 'vermelho'],
    'input_shape': [1, 4],
    'output_shape': [1, 3],
    'notes': 'Treinado com dados reais do Firestore + augmentação sintética'
}

meta_file = 'expiry_risk_meta.json'
with open(meta_file, 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print(json.dumps(meta, indent=2, ensure_ascii=False))
files.download(meta_file)
print(f"\n✅ Metadados baixados: {meta_file}")
print("   (coloque em assets/models/expiry_risk_meta.json)")